# LoKI application

The generic application of `ess.apps.panel` with the plots of `ess.instruments.loki`, on `local()`, over the runs of the LoKI detector test at Larmor.
It needs the extra `[loki-app]`.

`FolderFiles` serves the files of one folder, here the download cache of esssans' tutorial data.
The first cell downloads the files that are missing.
The table gives each file's metadata. The `sample` column names the sample of a sample run and of its transmission run.
Run 60393 is the can, and 60392 is its transmission run, also used as the empty beam, so its row names no role and a form offers it for both.

In [ ]:
from pathlib import Path

import panel as pn
from ess.loki import data

from ess.apps import FolderFiles, local
from ess.apps.panel import App
from ess.instruments.loki.bindings import BINDINGS
from ess.instruments.loki.plots import PLOTS

pn.extension('tabulator')

for download in (
    data.loki_tutorial_run_60392,
    data.loki_tutorial_background_run_60393,
    data.loki_tutorial_sample_run_60339,
    data.loki_tutorial_isis_polymer_sample_run,
    data.loki_tutorial_isis_polymer_transmission_run,
    data.loki_tutorial_agbeh_sample_run,
    data.loki_tutorial_agbeh_transmission_run,
    data.loki_tutorial_porous_silica_sample_run,
    data.loki_tutorial_porous_silica_transmission_run,
    data.loki_tutorial_deut_sds_sample_run,
    data.loki_tutorial_deut_sds_transmission_run,
    data.loki_tutorial_direct_beam_all_pixels,
):
    download()
mask_file = Path(data.loki_tutorial_mask_filenames()[0])
direct_beam_file = Path(data.loki_tutorial_direct_beam_all_pixels())

In [ ]:
def run(number: int, role: str | None = None, sample: str | None = None) -> dict[str, object]:
    row = {'file': f'{number}-2022-02-28_2215.nxs', 'run': number}
    if role is not None:
        row['role'] = role
    return row | ({'sample': sample} if sample else {})


table = [
    run(60339, 'sample'),
    run(60395, 'sample', 'ISIS polymer'),
    run(60394, 'transmission', 'ISIS polymer'),
    run(60387, 'sample', 'AgBeh'),
    run(60386, 'transmission', 'AgBeh'),
    run(60385, 'sample', 'porous silica'),
    run(60384, 'transmission', 'porous silica'),
    run(60389, 'sample', 'deut-SDS'),
    run(60388, 'transmission', 'deut-SDS'),
    run(60393, 'background'),
    run(60392),
    {'file': mask_file.name, 'kind': 'mask'},
    {'file': direct_beam_file.name, 'kind': 'calibration', 'role': 'direct_beam'},
]
files = FolderFiles(mask_file.parent, table)
client = local(proposal='loki-test', files=files, bind=BINDINGS)

## The order of use

The tab *Workflows* holds the forms, the tab *Outputs* the results.

1. **Pixel masks**: pick the form *Pixel masks*, give run 60339 and the mask file, and submit. In the tab *Outputs*, select the image and pick the view *Masks on image*.
2. **Beam centre**: pick the form *Beam centre*, give run 60339 and the masks as *From a record*, and submit. The view *Centre on image* marks the centre.
3. **I(Q)**: pick the form *I(Q)*, give a sample run and its transmission run, the can, its transmission run, the empty beam, the direct beam, the masks and the beam centre *From a record*, and the bins, and submit. The view *I(Q)* draws the result. The form keeps its values, so for the next sample change only the two sample fields.

In [ ]:
pn.Column(App(client, plots=PLOTS).panel(), height=800)